# INT315 Cluster Computing - Project
## Flight Delay Analytics and Airport Network Analysis using Apache Spark
Udayveer Singh Chaudhary · Reg. No. 12320106 · Roll No. 43 · Section K3P23RE

PySpark notebook: Spark SQL / DataFrames, RDD operations, shared variables, Structured Streaming demo, Spark ML feature pipeline, Linear Regression, Logistic Regression, Decision Tree, SVM, Naive Bayes, K-means, correlation and evaluation (confusion matrix, R2, RMSE, MAE).



In [1]:
import os
import sys

# Force Spark driver and workers to use the same Python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PySpark worker:", os.environ["PYSPARK_PYTHON"])

Python executable: /Users/mac/Downloads/INT315_project 6/.venv/bin/python
Python version: 3.11.18 (main, Oct 25 2026, 19:41:29) [Clang 17.0.0 (clang-1700.6.3.2)]
PySpark worker: /Users/mac/Downloads/INT315_project 6/.venv/bin/python


In [2]:
import os, json, time, shutil, math
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, TimestampType, LongType, StringType
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import Bucketizer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, LinearSVC, NaiveBayes
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.stat import Correlation

# This notebook must be opened from the INT315_project folder (VS Code and Jupyter use the notebook's folder).
OUT = "results"
os.makedirs(OUT, exist_ok=True)
R = {}                       # all numbers that go into the report
T0 = time.time()


def log(msg=""):
    print(msg, flush=True)


def timed(label, fn):
    t = time.time()
    res = fn()
    dt = round(time.time() - t, 2)
    log(f"   [time] {label}: {dt} s")
    return res, dt


from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FlightAnalytics")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Driver Python:", sys.executable)
print("Worker Python:", os.environ["PYSPARK_PYTHON"])

26/11/03 11:08:12 WARN Utils: Your hostname, macs-MacBook-Pro-4.local resolves to a loopback address: 127.0.0.1; using 10.36.76.53 instead (on interface en0)
26/11/03 11:08:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/03 11:08:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/03 11:08:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.3
Driver Python: /Users/mac/Downloads/INT315_project 6/.venv/bin/python
Worker Python: /Users/mac/Downloads/INT315_project 6/.venv/bin/python


In [3]:
# ================================================================ PART 1: load + clean
log("\nPART 1: Loading and cleaning the data (Spark SQL / DataFrame)")
raw = spark.read.parquet("data/flights-3m.parquet")
raw_n = raw.count()
log(f"Raw flights loaded: {raw_n:,}")
raw.printSchema()
bounds = raw.agg(F.min("date").alias("first"), F.max("date").alias("last"),
                 F.min("delay").alias("min_delay"), F.max("delay").alias("max_delay")).first()
log(f"Date range: {bounds['first']} to {bounds['last']} | delay range: {bounds['min_delay']} to {bounds['max_delay']}")

# keep realistic delays only (-60 .. 600 minutes); the rest are data errors / extreme outliers
clean = (raw.filter((F.col("delay") >= -60) & (F.col("delay") <= 600))
         .withColumn("hour", F.hour("date")).withColumn("dow", F.dayofweek("date"))
         .withColumn("dom", F.dayofmonth("date")).withColumn("month", F.month("date"))
         .withColumn("route", F.concat_ws("-", "origin", "destination")))
clean.cache()
clean_n = clean.count()
log(f"Flights after cleaning: {clean_n:,} (removed {raw_n - clean_n:,} extreme records)")
R.update(raw_rows=raw_n, clean_rows=clean_n, removed_rows=raw_n - clean_n,
         date_first=str(bounds["first"]), date_last=str(bounds["last"]),
         raw_min_delay=bounds["min_delay"], raw_max_delay=bounds["max_delay"],
         partitions_df=clean.rdd.getNumPartitions())

stats = clean.agg(F.avg("delay").alias("mean"), F.stddev("delay").alias("sd"),
                  F.avg((F.col("delay") > 15).cast("double")).alias("pct15"),
                  F.avg((F.col("delay") > 0).cast("double")).alias("pct0"),
                  F.avg("distance").alias("avg_dist")).first()
q = clean.approxQuantile("delay", [0.25, 0.5, 0.75, 0.9, 0.99], 0.001)
R.update(mean_delay=stats["mean"], sd_delay=stats["sd"], pct_delayed15=stats["pct15"],
         pct_late=stats["pct0"], avg_distance=stats["avg_dist"],
         q25=q[0], median=q[1], q75=q[2], q90=q[3], q99=q[4])
log(f"Mean delay = {stats['mean']:.2f} min | median = {q[1]} | 90th percentile = {q[3]} | "
    f"flights delayed more than 15 min = {stats['pct15']*100:.1f} %")
R["airports_n"] = clean.select("origin").distinct().count()
R["routes_n"] = clean.select("route").distinct().count()
log(f"Distinct origin airports = {R['airports_n']} | distinct routes = {R['routes_n']}")

# batch analytics with DataFrames -> csv files for the charts
def save(df, name):
    df.toPandas().to_csv(f"{OUT}/{name}.csv", index=False)

by_hour = clean.groupBy("hour").agg(F.count("*").alias("flights"), F.round(F.avg("delay"), 2).alias("avg_delay"),
                                    F.round(F.avg((F.col("delay") > 15).cast("double")) * 100, 2).alias("pct_delayed")).orderBy("hour")
by_dow = clean.groupBy("dow").agg(F.count("*").alias("flights"), F.round(F.avg("delay"), 2).alias("avg_delay"),
                                  F.round(F.avg((F.col("delay") > 15).cast("double")) * 100, 2).alias("pct_delayed")).orderBy("dow")
by_day = (clean.withColumn("day", F.to_date("date")).groupBy("day")
          .agg(F.count("*").alias("flights"), F.round(F.avg("delay"), 2).alias("avg_delay")).orderBy("day"))
by_origin = (clean.groupBy("origin").agg(F.count("*").alias("flights"), F.round(F.avg("delay"), 2).alias("avg_delay"),
                                         F.round(F.avg("distance"), 1).alias("avg_distance"),
                                         F.round(F.avg((F.col("delay") > 15).cast("double")), 4).alias("pct_delayed"))
             .orderBy(F.desc("flights")))
by_route = (clean.groupBy("route").agg(F.count("*").alias("flights"), F.round(F.avg("delay"), 2).alias("avg_delay"))
            .orderBy(F.desc("flights")))
for df, name in [(by_hour, "by_hour"), (by_dow, "by_dow"), (by_day, "by_day"), (by_origin, "by_origin"), (by_route.limit(30), "top_routes")]:
    save(df, name)
log("\nFlights and average delay by hour of the day:")
by_hour.show(24, False)
log("Top 5 busiest origin airports:")
by_origin.show(5, False)
log("Top 5 busiest routes:")
by_route.show(5, False)


PART 1: Loading and cleaning the data (Spark SQL / DataFrame)
Raw flights loaded: 3,000,000
root
 |-- date: timestamp_ntz (nullable = true)
 |-- delay: long (nullable = true)
 |-- distance: long (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)

Date range: 2001-01-01 00:01:00 to 2001-07-01 00:00:00 | delay range: -1116 to 1688
Flights after cleaning: 2,999,586 (removed 414 extreme records)
Mean delay = 6.59 min | median = -1.0 | 90th percentile = 35.0 | flights delayed more than 15 min = 20.0 %
Distinct origin airports = 229 | distinct routes = 3399

Flights and average delay by hour of the day:
+----+-------+---------+-----------+
|hour|flights|avg_delay|pct_delayed|
+----+-------+---------+-----------+
|0   |10338  |42.63    |45.83      |
|1   |6092   |28.27    |28.89      |
|2   |928    |86.02    |67.67      |
|3   |237    |98.6     |65.82      |
|4   |340    |13.14    |9.41       |
|5   |38427  |-4.01    |4.02       |
|6   |200746 

In [4]:
# ================================================================ PART 2: RDD operations
log("\nPART 2: RDD programming")
airports = pd.read_csv("data/airports.csv")
airport_list = [(r.iata, (r.name, r.city, r.state)) for r in airports.itertuples()]


PART 2: RDD programming


In [5]:
# 2.1 creating RDDs
nums = sc.parallelize(range(1, 1001), 4)
log(f"parallelize(range(1,1001), 4): partitions = {nums.getNumPartitions()}, sum = {nums.sum()}, "
    f"even numbers = {nums.filter(lambda x: x % 2 == 0).count()}")
flights = clean.select("origin", "destination", "delay", "distance").rdd.map(lambda r: (r[0], r[1], int(r[2]), int(r[3])))
flights_rdd = flights
log(f"flights RDD created from the DataFrame: partitions = {flights_rdd.getNumPartitions()}")
airport_rdd = sc.parallelize(airport_list)
log(f"airports RDD created with parallelize: {airport_rdd.count():,} airports")

parallelize(range(1,1001), 4): partitions = 4, sum = 500500, even numbers = 500
flights RDD created from the DataFrame: partitions = 4
airports RDD created with parallelize: 3,376 airports


In [6]:
# 2.2 transformations and actions
(r1, t1) = timed("count()", lambda: flights_rdd.count())
log(f"   action count() -> {r1:,} flights")
log(f"   action first() -> {flights_rdd.first()}")
log(f"   action take(3) -> {flights_rdd.take(3)}")
(total_dist, t2) = timed("reduce()", lambda: flights_rdd.map(lambda x: x[3]).reduce(lambda a, b: a + b))
log(f"   map + reduce: total distance flown = {total_dist:,} miles")
(delayed_n, t3) = timed("filter()", lambda: flights_rdd.filter(lambda x: x[2] > 15).count())
log(f"   filter: flights delayed more than 15 min = {delayed_n:,}")
(route_top, t4) = timed("map + reduceByKey + sortBy", lambda: flights_rdd.map(lambda x: (x[0] + "-" + x[1], 1))
                        .reduceByKey(lambda a, b: a + b).sortBy(lambda kv: -kv[1]).take(5))
log(f"   reduceByKey: busiest routes -> {route_top}")
(avg_origin, t5) = timed("average delay per origin with reduceByKey",
                         lambda: flights_rdd.map(lambda x: (x[0], (x[2], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                         .mapValues(lambda s: round(s[0] / s[1], 2)))
log(f"   mapValues: sample of average delay per origin -> {avg_origin.take(4)}")
cnt_by_dest = flights_rdd.map(lambda x: (x[1], 1)).countByKey()
log(f"   countByKey(): destinations = {len(cnt_by_dest)}, ATL arrivals = {cnt_by_dest.get('ATL')}")
words = sc.parallelize(["LAX-SFO", "SFO-LAX", "LAX-LAS"]).flatMap(lambda s: s.split("-"))
log(f"   flatMap on route names -> {words.collect()}, distinct -> {sorted(words.distinct().collect())}")
sample_pairs = flights_rdd.map(lambda x: (x[0], x[2])).sample(False, 0.0001, 7)
log(f"   sample(): {sample_pairs.count()} random flights, groupByKey example -> "
    f"{[(k, list(v)) for k, v in sample_pairs.groupByKey().take(2)]}")
joined = (avg_origin.join(airport_rdd.mapValues(lambda v: v[1])).takeOrdered(5, key=lambda kv: -kv[1][0]))
log(f"   join(): top 5 average delays with city names -> {joined}")

   [time] count(): 9.44 s
   action count() -> 2,999,586 flights
   action first() -> ('LAS', 'PHL', 33, 2176)
   action take(3) -> [('LAS', 'PHL', 33, 2176), ('ATL', 'SAV', 19, 215), ('MCI', 'MDW', 14, 405)]
   [time] reduce(): 13.01 s
   map + reduce: total distance flown = 2,194,274,522 miles
   [time] filter(): 8.9 s
   filter: flights delayed more than 15 min = 598,783
   [time] map + reduceByKey + sortBy: 9.33 s
   reduceByKey: busiest routes -> [('LAX-LAS', 8323), ('LAS-LAX', 8108), ('PHX-LAX', 7717), ('LAX-PHX', 7462), ('LAX-SAN', 6670)]
   [time] average delay per origin with reduceByKey: 0.02 s
   mapValues: sample of average delay per origin -> [('DTW', 5.0), ('PHX', 9.59), ('ORD', 9.24), ('MIA', 9.32)]
   countByKey(): destinations = 228, ATL arrivals = 124229
   flatMap on route names -> ['LAX', 'SFO', 'SFO', 'LAX', 'LAX', 'LAS'], distinct -> ['LAS', 'LAX', 'SFO']
   sample(): 287 random flights, groupByKey example -> [('TPA', [-10, 6, -5, 0, 3, -14, 7, -9]), ('MIA', [-12,

In [7]:
# 2.3 shared variables: broadcast + accumulator
state_of = sc.broadcast({iata: info[2] for iata, info in airport_list})
by_state = (flights_rdd.map(lambda x: (state_of.value.get(x[0], "??"), 1)).reduceByKey(lambda a, b: a + b)
            .sortBy(lambda kv: -kv[1]).take(5))
log(f"   broadcast variable (airport -> state lookup): flights by origin state -> {by_state}")
late_acc = sc.accumulator(0)
big_acc = sc.accumulator(0)


def count_late(rec):
    if rec[2] > 15:
        late_acc.add(1)
    if rec[2] > 180:
        big_acc.add(1)


flights_rdd.foreach(count_late)
log(f"   accumulators: delayed > 15 min = {late_acc.value:,}, delayed > 3 hours = {big_acc.value:,}")
hist_edges = list(range(-60, 190, 10))
edges, counts = clean.select("delay").rdd.map(lambda r: r[0]).histogram(hist_edges)
pd.DataFrame({"left": edges[:-1], "right": edges[1:], "flights": counts}).to_csv(f"{OUT}/delay_hist.csv", index=False)
R.update(rdd_count=r1, rdd_total_distance=total_dist, rdd_delayed15=delayed_n, rdd_top_routes=route_top,
         acc_late=late_acc.value, acc_big=big_acc.value, rdd_by_state=by_state,
         t_rdd_count=t1, t_rdd_reduce=t2, t_rdd_filter=t3, t_rdd_reducebykey=t4)

   broadcast variable (airport -> state lookup): flights by origin state -> [('CA', 370220), ('TX', 355891), ('FL', 202088), ('IL', 194286), ('NY', 134018)]
   accumulators: delayed > 15 min = 598,783, delayed > 3 hours = 13,890


In [8]:
# 2.4 in-memory computation: cache() versus recompute
log("\nIn-memory computation: repeated job with and without cache() on 1 million flights")
subset = clean.select("origin", "delay").sample(False, 0.34, 11).rdd.map(lambda r: (r[0], r[1]))


def job(rdd):
    t = time.time()
    rdd.map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).count()
    return round(time.time() - t, 2)


no_cache = [job(subset) for _ in range(4)]
cached = subset.persist(StorageLevel.MEMORY_ONLY)
with_cache = [job(cached) for _ in range(4)]
rows_cached = cached.count()
log(f"   rows in subset = {rows_cached:,}")
log(f"   without cache: {no_cache} s  | with cache: {with_cache} s")
cached.unpersist()
R["cache_benchmark"] = {"rows": rows_cached, "no_cache": no_cache, "with_cache": with_cache}


In-memory computation: repeated job with and without cache() on 1 million flights
   rows in subset = 1,019,603
   without cache: [3.37, 4.06, 5.26, 4.81] s  | with cache: [4.3, 1.21, 1.07, 0.96] s


In [9]:
# ================================================================ PART 3: streaming demo
log("\nPART 3: Batch versus real-time processing - Structured Streaming demo")
stream_dir = f"{OUT}/stream_input"
shutil.rmtree(stream_dir, ignore_errors=True)
os.makedirs(stream_dir)
chunk_pd = (clean.select("origin", "destination", "delay").sample(False, 0.08, 5).limit(200000).toPandas())
for i in range(10):
    chunk_pd.iloc[i * 20000:(i + 1) * 20000].to_csv(f"{stream_dir}/part-{i:02d}.csv", index=False, header=False)
schema = StructType([StructField("origin", StringType()), StructField("destination", StringType()), StructField("delay", LongType())])
stream_df = spark.readStream.schema(schema).option("maxFilesPerTrigger", 1).csv(stream_dir)
agg = stream_df.groupBy("origin").agg(F.count("*").alias("flights"), F.round(F.avg("delay"), 2).alias("avg_delay"))
query = agg.writeStream.outputMode("complete").format("memory").queryName("origin_live").start()
query.processAllAvailable()
progress = query.recentProgress
batches = [p["numInputRows"] for p in progress if p["numInputRows"] > 0]
live_top = spark.sql("select * from origin_live order by flights desc limit 5").toPandas()
query.stop()
log(f"   micro-batches processed: {len(batches)} | rows per micro-batch: {batches}")
log("   running result after the last micro-batch (top 5 origins):")
log(live_top.to_string(index=False))
R["stream"] = {"micro_batches": len(batches), "rows_per_batch": batches,
               "top": live_top.to_dict(orient="records")}


PART 3: Batch versus real-time processing - Structured Streaming demo
   micro-batches processed: 10 | rows per micro-batch: [20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000, 20000]
   running result after the last micro-batch (top 5 origins):
origin  flights  avg_delay
   ORD    10966       8.23
   DFW    10452       7.10
   ATL     8170       7.32
   LAX     7636       7.44
   PHX     6162       9.90


In [10]:
# ================================================================ PART 4: feature extraction + transformation
log("\nPART 4: Feature extraction and transformation (Spark ML pipeline)")
origin_cnt = clean.groupBy("origin").agg(F.count("*").alias("origin_flights"))
dest_cnt = clean.groupBy("destination").agg(F.count("*").alias("dest_flights"))
feat_df = (clean.join(origin_cnt, "origin").join(dest_cnt, "destination")
           .withColumn("log_distance", F.log1p("distance")).withColumn("dow_idx", (F.col("dow") - 1).cast("double"))
           .withColumn("label_reg", F.col("delay").cast("double"))
           .withColumn("label_cls", (F.col("delay") > 15).cast("double"))
           .select("origin", "destination", "hour", "dow", "dow_idx", "dom", "month", "distance", "log_distance",
                   "origin_flights", "dest_flights", "delay", "label_reg", "label_cls"))
num_cols = ["hour", "dom", "month", "distance", "log_distance", "origin_flights", "dest_flights"]
pipe = Pipeline(stages=[
    Bucketizer(splits=[0, 500, 1500, float("inf")], inputCol="distance", outputCol="band"),
    OneHotEncoder(inputCols=["band", "dow_idx"], outputCols=["band_vec", "dow_vec"]),
    VectorAssembler(inputCols=num_cols + ["band_vec", "dow_vec"], outputCol="raw_features"),
    StandardScaler(inputCol="raw_features", outputCol="features", withMean=False, withStd=True),
    MinMaxScaler(inputCol="raw_features", outputCol="features_mm"),
])
feature_names = num_cols + ["band_short", "band_medium"] + ["dow_Sun", "dow_Mon", "dow_Tue", "dow_Wed", "dow_Thu", "dow_Fri"]
sample_frac = 0.2
ml_df = feat_df.sample(False, sample_frac, 42)
pipe_model = pipe.fit(ml_df)
ml_ready = pipe_model.transform(ml_df).select("label_reg", "label_cls", "features", "features_mm", "origin", "delay", "hour", "dow", "dom", "month", "distance", "origin_flights", "dest_flights")
train, test = ml_ready.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()
n_train, n_test = train.count(), test.count()
log(f"   stages: Bucketizer -> OneHotEncoder -> VectorAssembler -> StandardScaler / MinMaxScaler")
log(f"   feature vector length = {len(feature_names)} | sample = {sample_frac*100:.0f} % of the clean data | train = {n_train:,} | test = {n_test:,}")
R.update(feature_names=feature_names, n_train=n_train, n_test=n_test, sample_frac=sample_frac)

# correlation matrix (Pearson) for the heat map
corr_cols = ["delay", "distance", "hour", "dow", "dom", "month", "origin_flights", "dest_flights"]
vec = VectorAssembler(inputCols=corr_cols, outputCol="corr_vec").transform(ml_ready.select(*corr_cols))
corr = Correlation.corr(vec, "corr_vec", "pearson").head()[0].toArray()
pd.DataFrame(corr, index=corr_cols, columns=corr_cols).to_csv(f"{OUT}/correlation.csv")
log("   Pearson correlation of delay with the other columns: " +
    ", ".join(f"{c}={corr[0][i]:.3f}" for i, c in enumerate(corr_cols) if i > 0))
R["corr_with_delay"] = {c: round(float(corr[0][i]), 4) for i, c in enumerate(corr_cols) if i > 0}


PART 4: Feature extraction and transformation (Spark ML pipeline)
   stages: Bucketizer -> OneHotEncoder -> VectorAssembler -> StandardScaler / MinMaxScaler
   feature vector length = 15 | sample = 20 % of the clean data | train = 480,282 | test = 120,052
   Pearson correlation of delay with the other columns: distance=-0.015, hour=0.162, dow=0.025, dom=-0.006, month=-0.010, origin_flights=0.032, dest_flights=0.002


In [11]:
# ================================================================ PART 5: machine learning
log("\nPART 5: Spark ML models (pyspark.ml)")
# ---- regression
mean_train = train.agg(F.avg("label_reg")).first()[0]
base_rmse = test.agg(F.sqrt(F.avg((F.col("label_reg") - mean_train) ** 2))).first()[0]
base_mae = test.agg(F.avg(F.abs(F.col("label_reg") - mean_train))).first()[0]
reg_eval = {m: RegressionEvaluator(labelCol="label_reg", predictionCol="prediction", metricName=m) for m in ["rmse", "mae", "r2"]}
reg_results = {"Baseline (mean)": {"rmse": base_rmse, "mae": base_mae, "r2": 0.0, "train_time": 0.0}}
log(f"\nRegression target: delay (minutes). Baseline that always predicts the mean ({mean_train:.2f}): RMSE = {base_rmse:.3f}, MAE = {base_mae:.3f}")
for name, est in [("Linear Regression", LinearRegression(featuresCol="features", labelCol="label_reg", maxIter=50)),
                  ("Decision Tree Regressor", DecisionTreeRegressor(featuresCol="features", labelCol="label_reg", maxDepth=6, seed=42))]:
    model, dt = timed(f"train {name}", lambda e=est: e.fit(train))
    pred = model.transform(test)
    res = {m: float(ev.evaluate(pred)) for m, ev in reg_eval.items()}
    res["train_time"] = dt
    reg_results[name] = res
    log(f"   {name}: RMSE = {res['rmse']:.3f} | MAE = {res['mae']:.3f} | R2 = {res['r2']:.4f}")
    if name == "Linear Regression":
        R["linreg_coef"] = dict(zip(feature_names, [round(float(c), 4) for c in model.coefficients]))
    else:
        imp = model.featureImportances.toArray()
        pd.DataFrame({"feature": feature_names, "importance": imp}).sort_values("importance", ascending=False).to_csv(f"{OUT}/dt_importance.csv", index=False)
        R["dt_top_features"] = sorted(zip(feature_names, [round(float(v), 4) for v in imp]), key=lambda t: -t[1])[:5]
R["regression"] = reg_results
# sample for the actual-vs-predicted plot
lr_model = LinearRegression(featuresCol="features", labelCol="label_reg", maxIter=50).fit(train)
lr_model.transform(test).select("label_reg", "prediction").sample(False, 0.01, 3).toPandas().to_csv(f"{OUT}/reg_scatter.csv", index=False)

# ---- classification
pos_rate = test.agg(F.avg("label_cls")).first()[0]
majority = max(pos_rate, 1 - pos_rate)
log(f"\nClassification target: delayed more than 15 minutes (1) or not (0). Share of delayed flights in the test set = {pos_rate*100:.2f} %"
    f" -> a model that always says 'on time' is {majority*100:.2f} % accurate")
auc_eval = BinaryClassificationEvaluator(labelCol="label_cls", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
n_pos = train.filter("label_cls = 1").count()
w1 = (n_train - n_pos) / n_pos            # class weight so that delayed flights count as much as on-time flights
train_w = train.withColumn("weight", F.when(F.col("label_cls") == 1.0, F.lit(w1)).otherwise(F.lit(1.0))).cache()
log(f"   delayed flights are rare, so a class weight of {w1:.2f} is given to the delayed class (balanced training)")
R["class_weight"] = w1
cls_models = [
    ("Logistic Regression (no weights)", LogisticRegression(featuresCol="features", labelCol="label_cls", maxIter=50), False),
    ("Logistic Regression", LogisticRegression(featuresCol="features", labelCol="label_cls", maxIter=50, weightCol="weight"), True),
    ("Decision Tree", DecisionTreeClassifier(featuresCol="features", labelCol="label_cls", maxDepth=6, seed=42, weightCol="weight"), True),
    ("SVM (LinearSVC)", LinearSVC(featuresCol="features", labelCol="label_cls", maxIter=30, regParam=0.01, weightCol="weight"), True),
    ("Naive Bayes", NaiveBayes(featuresCol="features_mm", labelCol="label_cls", modelType="multinomial", weightCol="weight"), True),
]
cls_results = {}
conf = {}
for name, est, weighted in cls_models:
    model, dt = timed(f"train {name}", lambda e=est, w=weighted: e.fit(train_w if w else train))
    pred = model.transform(test)
    cm = {(int(r["label_cls"]), int(r["prediction"])): r["count"] for r in pred.groupBy("label_cls", "prediction").count().collect()}
    tn, fp, fn, tp = cm.get((0, 0), 0), cm.get((0, 1), 0), cm.get((1, 0), 0), cm.get((1, 1), 0)
    total = tn + fp + fn + tp
    acc = (tn + tp) / total
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    auc = float(auc_eval.evaluate(pred))
    cls_results[name] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc, "train_time": dt}
    conf[name] = [[tn, fp], [fn, tp]]
    log(f"   {name}: accuracy = {acc:.4f} | precision = {prec:.4f} | recall = {rec:.4f} | F1 = {f1:.4f} | AUC = {auc:.4f}")
    log(f"      confusion matrix [[TN, FP], [FN, TP]] = {[[tn, fp], [fn, tp]]}")
R["classification"] = cls_results
R["confusion"] = conf
R.update(test_pos_rate=pos_rate, majority_accuracy=majority)

# ---- K-means clustering of airports
log("\nK-means clustering of airports (unsupervised)")
ap = (clean.groupBy("origin").agg(F.count("*").alias("flights"), F.avg("delay").alias("avg_delay"),
                                  F.avg("distance").alias("avg_distance"), F.avg((F.col("delay") > 15).cast("double")).alias("pct_delayed"))
      .filter(F.col("flights") >= 1000).withColumn("log_flights", F.log10("flights")))
ap_vec = VectorAssembler(inputCols=["log_flights", "avg_delay", "avg_distance", "pct_delayed"], outputCol="raw").transform(ap)
ap_scaled = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True).fit(ap_vec).transform(ap_vec).cache()
log(f"   airports with at least 1,000 flights: {ap_scaled.count()}")
sil = {}
cost = {}
sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
for k in range(2, 9):
    km = KMeans(k=k, seed=42, maxIter=50, featuresCol="features").fit(ap_scaled)
    out = km.transform(ap_scaled)
    sil[k] = round(float(sil_eval.evaluate(out)), 4)
    cost[k] = round(float(km.summary.trainingCost), 3)
log(f"   silhouette for k=2..8: {sil}")
best_k = max(sil, key=sil.get)
if best_k == 2:                       # two groups are too coarse for interpretation, use the best k from 3 to 8
    best_k = max((k for k in sil if k >= 3), key=lambda k: sil[k])
    log(f"   k = 2 has the highest silhouette, but at least 3 groups are needed for interpretation -> using k = {best_k}")
km = KMeans(k=best_k, seed=42, maxIter=50, featuresCol="features").fit(ap_scaled)
clustered = km.transform(ap_scaled).select("origin", "flights", "avg_delay", "avg_distance", "pct_delayed", F.col("prediction").alias("cluster"))
cp = clustered.toPandas().merge(airports[["iata", "name", "city", "state", "latitude", "longitude"]], left_on="origin", right_on="iata", how="left")
cp.to_csv(f"{OUT}/kmeans_airports.csv", index=False)
summary = cp.groupby("cluster").agg(airports=("origin", "count"), flights=("flights", "mean"), avg_delay=("avg_delay", "mean"),
                                    avg_distance=("avg_distance", "mean"), pct_delayed=("pct_delayed", "mean")).round(3)
log(f"   chosen k = {best_k}; cluster profile:")
log(summary.to_string())
R["kmeans"] = {"silhouette": sil, "cost": cost, "k": best_k, "profile": summary.reset_index().to_dict(orient="records"),
               "members": {int(c): cp[cp.cluster == c].sort_values("flights", ascending=False).origin.head(8).tolist() for c in sorted(cp.cluster.unique())}}

R["total_runtime_s"] = round(time.time() - T0, 1)
with open(f"{OUT}/metrics.json", "w") as fh:
    json.dump(R, fh, indent=2, default=str)
log(f"\nAll results saved in the folder '{OUT}'. Total run time: {R['total_runtime_s']} s")
spark.stop()


PART 5: Spark ML models (pyspark.ml)

Regression target: delay (minutes). Baseline that always predicts the mean (6.62): RMSE = 30.839, MAE = 18.452
   [time] train Linear Regression: 2.19 s
   Linear Regression: RMSE = 30.345 | MAE = 18.187 | R2 = 0.0318
   [time] train Decision Tree Regressor: 2.76 s
   Decision Tree Regressor: RMSE = 29.640 | MAE = 17.886 | R2 = 0.0762

Classification target: delayed more than 15 minutes (1) or not (0). Share of delayed flights in the test set = 20.06 % -> a model that always says 'on time' is 79.94 % accurate
   delayed flights are rare, so a class weight of 4.00 is given to the delayed class (balanced training)
   [time] train Logistic Regression (no weights): 4.48 s
   Logistic Regression (no weights): accuracy = 0.7994 | precision = 0.3636 | recall = 0.0002 | F1 = 0.0003 | AUC = 0.6321
      confusion matrix [[TN, FP], [FN, TP]] = [[95964, 7], [24077, 4]]
   [time] train Logistic Regression: 4.83 s
   Logistic Regression: accuracy = 0.5924 | pr